# Clustering V1 — Apartment Complex Segmentation

## 목적

서울 아파트 단지를 가격 자체가 아닌
**주택 구조 + 입지 특성**을 기준으로 유형화한다.

군집화의 목적은 예측이 아니라 다음과 같다.

- 비슷한 특성을 가진 아파트 단지 탐색
- 단지 유형별 특성 비교
- Regression 결과의 해석 보조
- 웹 서비스에서 단지 유형 설명

## 기본 원칙

- 단위: 거래가 아닌 **아파트 단지**
- 학습기간: 2021-07 ~ 2025-12
- Test 기간: 2026-01 ~ 2026-06
- 가격(`거래금액`, `평당가`)은 군집 Feature에서 제외
- 가격은 최종 군집을 만든 후 사후 해석에만 사용
- 거리 Feature는 log1p 변환 검토
- Feature Scaling 적용
- KMeans / GMM / Agglomerative 비교

## 평가 기준

1. Silhouette Score — 높을수록 좋음
2. Davies-Bouldin Index — 낮을수록 좋음
3. Calinski-Harabasz Score — 높을수록 좋음
4. 군집 크기 균형
5. 군집별 Silhouette
6. 비즈니스 해석 가능성

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
)


plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


PROJECT_DIR = Path.cwd().parents[3]

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
    / "df_final_v1.parquet"
)

df = pd.read_parquet(DATA_PATH)

df["거래월일자"] = pd.to_datetime(
    df["거래월일자"]
)

print("전체 데이터:", df.shape)
print(
    "기간:",
    df["거래월일자"].min(),
    "~",
    df["거래월일자"].max(),
)

전체 데이터: (234302, 124)
기간: 2021-07-01 00:00:00 ~ 2026-06-01 00:00:00


In [2]:
# ============================================================
# Clustering Train / Test
# ============================================================

cluster_train_df = df[
    df["거래월일자"] < "2026-01-01"
].copy()

cluster_test_df = df[
    df["거래월일자"] >= "2026-01-01"
].copy()


print(
    "Train 거래:",
    f"{len(cluster_train_df):,}",
)

print(
    "Test 거래:",
    f"{len(cluster_test_df):,}",
)

print(
    "Train 단지:",
    f"{cluster_train_df['단지일련번호'].nunique():,}",
)

print(
    "Test 단지:",
    f"{cluster_test_df['단지일련번호'].nunique():,}",
)

Train 거래: 194,733
Test 거래: 39,569
Train 단지: 7,581
Test 단지: 5,038


## 시간 분리 기준

군집화는 비지도학습이지만,
2026년 데이터를 이용해 미리 단지 유형을 결정하지 않도록
2025년 12월까지의 데이터만 사용해 군집 구조를 학습한다.

2026년 상반기 데이터는 다음 용도로만 사용한다.

- 기존 단지 군집 연결 Coverage 확인
- 신규 단지 일반화 가능성 확인
- Regression V1의 군집별 성능 분석

이를 통해 Clustering도 Regression / Classification과 동일하게
시간 기준을 명확하게 유지한다.

## 1. 단지 단위 데이터 구성

거래 단위 데이터를 단지 단위로 집계한다.

- 면적·연식·층·거리 특성은 단지별 중앙값 사용
- `전용면적_IQR`로 단지 내 평형 구성 다양성을 반영
- 교육 접근성은 V1에서 유치원 + 초등학교만 결합
- 중학교·고등학교 거리는 최종 군집 Feature에서 제외

In [3]:
# ============================================================
# 거래 단위 → 단지 단위
# ============================================================

complex_df = (
    cluster_train_df
    .groupby("단지일련번호")
    .agg(
        단지명=("단지명", "first"),
        법정동=("법정동", "first"),

        거래건수=("transaction_id", "size"),

        전용면적=("전용면적", "median"),
        전용면적_Q25=("전용면적", lambda x: x.quantile(0.25)),
        전용면적_Q75=("전용면적", lambda x: x.quantile(0.75)),

        건축년도=("건축년도", "median"),
        층=("층", "median"),

        유치원거리_m=("유치원거리_m", "median"),
        초등학교거리_m=("초등학교거리_m", "median"),
        중학교거리_m=("중학교거리_m", "median"),
        고등학교거리_m=("고등학교거리_m", "median"),

        역세권거리_m=("역세권거리_m", "median"),
        응급의료기관거리_m=("응급의료기관거리_m", "median"),
        대규모점포거리_m=("대규모점포거리_m", "median"),
        주요일자리거리_m=("주요일자리거리_m", "median"),
    )
    .reset_index()
)


# 2025년 말 기준 연식
complex_df["기준연식"] = (
    2025 - complex_df["건축년도"]
).clip(lower=0)


# 단지 내 면적 구성 다양성
complex_df["전용면적_IQR"] = (
    complex_df["전용면적_Q75"]
    - complex_df["전용면적_Q25"]
)


# 유치원 + 초등학교 접근성
complex_df["유초등접근거리_m"] = (
    complex_df[
        [
            "유치원거리_m",
            "초등학교거리_m",
        ]
    ]
    .mean(axis=1)
)


print("군집 대상 단지:", f"{len(complex_df):,}")

군집 대상 단지: 7,581


In [4]:
cluster_feature_sets = {

    "A_주택구조": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",
    ],

    "B_교육생활": [
        "유초등접근거리_m",
        "대규모점포거리_m",
        "응급의료기관거리_m",
    ],

    "C_교통업무": [
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "D_핵심통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "유초등접근거리_m",
        "역세권거리_m",
        "주요일자리거리_m",
    ],

    "E_전체통합": [
        "전용면적",
        "전용면적_IQR",
        "기준연식",
        "층",

        "유초등접근거리_m",
        "역세권거리_m",

        "대규모점포거리_m",
        "응급의료기관거리_m",
        "주요일자리거리_m",
    ],
}

In [5]:
# ============================================================
# Clustering 전처리
# ============================================================

LOG_COLS = {
    "전용면적",
    "전용면적_IQR",
    "유초등접근거리_m",
    "역세권거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
    "주요일자리거리_m",
}

def prepare_cluster_data(data, feature_cols):

    model_df = data[feature_cols].copy()

    # 음수 방지
    for col in feature_cols:
        if col in LOG_COLS:
            model_df[col] = np.log1p(
                model_df[col].clip(lower=0)
            )

    scaler = StandardScaler()

    X = scaler.fit_transform(
        model_df
    )

    return X, scaler

## 2. Feature Set · 알고리즘 · K 비교

V1에서는 다음 Feature Set을 비교한다.

- **A_주택구조**: 면적, 면적 다양성, 연식, 층
- **B_교육생활**: 유·초등 접근성, 대규모점포, 응급의료기관
- **C_교통업무**: 역세권, 주요 일자리 접근성
- **D_핵심통합 / E_전체통합**: 구조와 입지 특성을 결합한 실험군

알고리즘은 KMeans / GMM / Agglomerative를 비교하고 K=2~6을 탐색한다.

Screening 단계의 Silhouette는 계산량을 줄이기 위해 최대 3,000개 표본으로 계산한다.
최종 모델은 전체 단지를 사용해 다시 평가한다.

In [6]:
# ============================================================
# KMeans / GMM / Agglomerative 비교
# ============================================================

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)


comparison_results = []


for feature_set_name, feature_cols in cluster_feature_sets.items():

    X, _ = prepare_cluster_data(
        complex_df,
        feature_cols,
    )


    for k in range(2, 7):

        # ----------------------------------------------------
        # KMeans
        # ----------------------------------------------------

        kmeans = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=42,
        )

        kmeans_labels = kmeans.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "KMeans",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                kmeans_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                kmeans_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                kmeans_labels,
            ),
        })


        # ----------------------------------------------------
        # GMM
        # ----------------------------------------------------

        gmm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=42,
            n_init=3,
        )

        gmm_labels = gmm.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "GMM",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                gmm_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                gmm_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                gmm_labels,
            ),
        })


        # ----------------------------------------------------
        # Agglomerative
        # ----------------------------------------------------

        agg = AgglomerativeClustering(
            n_clusters=k,
            linkage="ward",
        )

        agg_labels = agg.fit_predict(X)


        comparison_results.append({
            "Feature_Set": feature_set_name,
            "Algorithm": "Agglomerative",
            "K": k,

            "Silhouette": silhouette_score(
                X,
                agg_labels,
                sample_size=min(3000, len(X)),
                random_state=42,
            ),

            "Davies_Bouldin": davies_bouldin_score(
                X,
                agg_labels,
            ),

            "Calinski_Harabasz": calinski_harabasz_score(
                X,
                agg_labels,
            ),
        })


cluster_comparison_df = pd.DataFrame(
    comparison_results
)


display(
    cluster_comparison_df
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .head(30)
    .round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
16,B_교육생활,GMM,2,0.5010,1.4241,1206.6895
31,C_교통업무,GMM,2,0.3860,1.1802,4073.2876
30,C_교통업무,KMeans,2,0.3774,1.1384,4394.6136
33,C_교통업무,KMeans,3,0.3671,0.9032,5158.0597
42,C_교통업무,KMeans,6,0.3426,0.8578,4947.5662
39,C_교통업무,KMeans,5,0.3370,0.8909,4969.2898
35,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
3,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
9,A_주택구조,KMeans,5,0.3305,1.1462,2469.6637
43,C_교통업무,GMM,6,0.3279,0.8944,4456.0920


In [7]:
# ============================================================
# 각 알고리즘 / Feature Set별 최고 조합
# ============================================================

best_cluster_candidates = (
    cluster_comparison_df
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .groupby(
        [
            "Feature_Set",
            "Algorithm",
        ],
        as_index=False,
    )
    .first()
    .sort_values(
        "Silhouette",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    best_cluster_candidates.round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
0,B_교육생활,GMM,2,0.5010,1.4241,1206.6895
1,C_교통업무,GMM,2,0.3860,1.1802,4073.2876
2,C_교통업무,KMeans,2,0.3774,1.1384,4394.6136
3,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
4,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
5,E_전체통합,Agglomerative,2,0.2951,1.8441,646.0881
6,A_주택구조,Agglomerative,6,0.2873,1.2681,1936.9557
7,B_교육생활,KMeans,4,0.2719,0.9905,2958.0576
8,B_교육생활,Agglomerative,3,0.2272,1.2051,2011.2716
9,D_핵심통합,KMeans,4,0.1968,1.6122,1351.7109


In [8]:
k3_plus_candidates = (
    cluster_comparison_df[
        cluster_comparison_df["K"] >= 3
    ]
    .sort_values(
        [
            "Silhouette",
            "Davies_Bouldin",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    k3_plus_candidates
    .head(20)
    .round(4)
)

,Feature_Set,Algorithm,K,Silhouette,Davies_Bouldin,Calinski_Harabasz
0,C_교통업무,KMeans,3,0.3671,0.9032,5158.0597
1,C_교통업무,KMeans,6,0.3426,0.8578,4947.5662
2,C_교통업무,KMeans,5,0.3370,0.8909,4969.2898
3,C_교통업무,Agglomerative,3,0.3349,1.0024,4273.9584
4,A_주택구조,KMeans,3,0.3320,1.1818,2786.1853
5,A_주택구조,KMeans,5,0.3305,1.1462,2469.6637
6,C_교통업무,GMM,6,0.3279,0.8944,4456.0920
7,C_교통업무,Agglomerative,4,0.3251,1.0692,4097.6721
8,C_교통업무,GMM,3,0.3220,0.9429,4052.5380
9,C_교통업무,KMeans,4,0.3216,0.9581,4833.4308


## 모델 선택 해석

### 1) 최고 Silhouette만으로 최종 모델을 선택하지 않음

`B_교육생활 + GMM + K=2`는 Screening Silhouette가 **0.5010**으로 가장 높았지만,
군집 비율이 약 **94.5% : 5.5%**로 크게 불균형했다.
소수 군집의 평균 Silhouette도 낮고 음수 비율이 높아,
서울 아파트 전체를 설명하는 실용적인 세그멘테이션으로 보기 어렵다고 판단했다.

### 2) K=2는 참고 결과로 남기고 최종 V1은 K=3 사용

K=2는 수치상 분리도가 높았지만 서비스에서 단지 유형을
사실상 두 집단으로만 나누는 구조가 되어 해석력이 제한적이었다.

따라서 V1에서는 **K≥3 후보 중 분리도·안정성·해석 가능성을 함께 고려**했다.

### 3) 최종 V1 구조

| 구분 | Feature | Algorithm | K | Screening Silhouette |
|---|---|---|---:|---:|
| 입지 유형 | 역세권거리, 주요일자리거리 | KMeans | 3 | 0.3671 |
| 주택특성 유형 | 전용면적, 전용면적 IQR, 연식, 층 | KMeans | 3 | 0.3320 |

하나의 통합 군집보다 **입지와 주택특성을 독립적인 두 축으로 분리**했을 때
각 군집의 의미가 더 명확하고 서비스 활용성도 높았다.

## 3. 최종 입지 Segmentation

입지 군집은 **역세권 접근성 + 주요 일자리 접근성** 두 축으로 구성한다.

주택면적·연식·생활시설 거리는 군집 생성에 사용하지 않고
군집이 만들어진 뒤 **사후 해석용 변수**로만 활용한다.

In [9]:
# ============================================================
# 최종 입지 군집 - KMeans K=3
# ============================================================

LOCATION_FEATURES = [
    "역세권거리_m",
    "주요일자리거리_m",
]

LOCATION_K = 3

X_location, location_scaler = prepare_cluster_data(
    complex_df,
    LOCATION_FEATURES,
)

location_model = KMeans(
    n_clusters=LOCATION_K,
    n_init=20,
    random_state=42,
)

location_labels = location_model.fit_predict(
    X_location
)

complex_df["location_cluster"] = location_labels

print(
    "Silhouette:",
    f"{silhouette_score(X_location, location_labels):.4f}",
)
print(
    "Davies-Bouldin:",
    f"{davies_bouldin_score(X_location, location_labels):.4f}",
)
print(
    "Calinski-Harabasz:",
    f"{calinski_harabasz_score(X_location, location_labels):.2f}",
)

Silhouette: 0.3725
Davies-Bouldin: 0.9032
Calinski-Harabasz: 5158.06


In [10]:
location_cluster_size = (
    complex_df["location_cluster"]
    .value_counts()
    .sort_index()
    .rename("단지수")
    .to_frame()
)

location_cluster_size["단지비율_%"] = (
    location_cluster_size["단지수"]
    / len(complex_df)
    * 100
)

display(
    location_cluster_size.round(2)
)

,단지수,단지비율_%
location_cluster,,
0,2185,28.82
1,1687,22.25
2,3709,48.92


In [11]:
location_silhouette_values = silhouette_samples(
    X_location,
    location_labels,
)

location_silhouette_results = []

for cluster in sorted(
    complex_df["location_cluster"].unique()
):
    mask = (
        complex_df["location_cluster"].to_numpy()
        == cluster
    )

    location_silhouette_results.append({
        "cluster": cluster,
        "단지수": int(mask.sum()),
        "평균_Silhouette": location_silhouette_values[mask].mean(),
        "음수비율_%": (
            location_silhouette_values[mask] < 0
        ).mean() * 100,
    })

location_silhouette_df = pd.DataFrame(
    location_silhouette_results
)

display(
    location_silhouette_df.round(3)
)

,cluster,단지수,평균_Silhouette,음수비율_%
0,0,2185,0.340,1.281
1,1,1687,0.309,5.572
2,2,3709,0.420,0.000


In [12]:
location_profile_cols = [
    "역세권거리_m",
    "주요일자리거리_m",

    # 해석용
    "전용면적",
    "기준연식",
    "층",
    "유초등접근거리_m",
    "대규모점포거리_m",
    "응급의료기관거리_m",
]

location_cluster_profile = (
    complex_df
    .groupby("location_cluster")[location_profile_cols]
    .median()
)

location_cluster_profile["단지수"] = (
    complex_df
    .groupby("location_cluster")
    .size()
)

location_cluster_profile["단지비율_%"] = (
    location_cluster_profile["단지수"]
    / len(complex_df)
    * 100
)

display(
    location_cluster_profile.round(2)
)

,역세권거리_m,주요일자리거리_m,전용면적,기준연식,층,유초등접근거리_m,대규모점포거리_m,응급의료기관거리_m,단지수,단지비율_%
location_cluster,,,,,,,,,,
0,266.14,5915.04,76.17,21.0,7.0,322.85,430.95,1047.98,2185,28.82
1,390.62,1522.48,81.84,22.0,6.0,342.64,534.24,888.61,1687,22.25
2,659.26,5766.52,81.95,22.0,6.0,319.17,628.33,1173.56,3709,48.92


### 입지 군집 해석

최종 전체 단지 기준 Silhouette는 **0.3725**이며,
Screening 단계의 0.3671과 차이가 나는 이유는 Screening에서는 최대 3,000개 표본,
최종 평가는 7,581개 전체 단지를 사용했기 때문이다.

| Cluster | 단지 비율 | 역세권 중앙값 | 주요 일자리 중앙값 | 해석 |
|---|---:|---:|---:|---|
| 0 | 28.82% | 266m | 5.92km | **역세권 생활주거형** |
| 1 | 22.25% | 391m | 1.52km | **직주근접 도심형** |
| 2 | 48.92% | 659m | 5.77km | **일반 주거형** |

- **Cluster 0**: 지하철 접근성이 가장 뛰어나지만 주요 업무지구와는 거리가 있어 생활형 역세권 주거지로 해석한다.
- **Cluster 1**: 주요 일자리 접근성이 압도적으로 좋고 역 접근성도 양호해 직주근접 도심형으로 해석한다.
- **Cluster 2**: 역세권·업무지 접근성이 상대적으로 낮고 가장 큰 비중을 차지해 일반 주거형으로 해석한다.

군집별 평균 Silhouette는 약 **0.309~0.420**이며,
가장 불안정한 Cluster 1도 음수 비율이 약 **5.6%** 수준으로 큰 쏠림 없이 유지된다.

In [13]:
location_cluster_name_map = {
    0: "역세권 생활주거형",
    1: "직주근접 도심형",
    2: "일반 주거형",
}

complex_df["location_cluster_name"] = (
    complex_df["location_cluster"]
    .map(location_cluster_name_map)
)

## 4. 최종 주택특성 Segmentation

주택특성 군집은 입지와 분리해
**면적 + 평형 구성 다양성 + 연식 + 층**으로 단지의 구조적 성격을 유형화한다.

입지 Feature는 주택특성 군집에 포함하지 않는다.

In [14]:
# ============================================================
# 주택특성 군집 - KMeans K=3
# ============================================================

HOUSING_FEATURES = [
    "전용면적",
    "전용면적_IQR",
    "기준연식",
    "층",
]

HOUSING_K = 3


X_housing, housing_scaler = prepare_cluster_data(
    complex_df,
    HOUSING_FEATURES,
)


housing_model = KMeans(
    n_clusters=HOUSING_K,
    n_init=20,
    random_state=42,
)


housing_labels = housing_model.fit_predict(
    X_housing
)


complex_df["housing_cluster"] = housing_labels


print(
    "Silhouette:",
    f"{silhouette_score(X_housing, housing_labels):.4f}",
)

print(
    "Davies-Bouldin:",
    f"{davies_bouldin_score(X_housing, housing_labels):.4f}",
)

print(
    "Calinski-Harabasz:",
    f"{calinski_harabasz_score(X_housing, housing_labels):.2f}",
)

Silhouette: 0.3320
Davies-Bouldin: 1.1818
Calinski-Harabasz: 2786.19


In [15]:
housing_cluster_size = (
    complex_df["housing_cluster"]
    .value_counts()
    .sort_index()
    .rename("단지수")
    .to_frame()
)

housing_cluster_size["단지비율_%"] = (
    housing_cluster_size["단지수"]
    / len(complex_df)
    * 100
)

display(
    housing_cluster_size.round(2)
)

,단지수,단지비율_%
housing_cluster,,
0,2757,36.37
1,3783,49.90
2,1041,13.73


In [16]:
housing_silhouette_values = silhouette_samples(
    X_housing,
    housing_labels,
)

housing_silhouette_results = []


for cluster in sorted(
    complex_df["housing_cluster"].unique()
):

    mask = (
        complex_df["housing_cluster"].to_numpy()
        == cluster
    )

    housing_silhouette_results.append({
        "cluster": cluster,
        "단지수": int(mask.sum()),
        "단지비율_%": mask.mean() * 100,
        "평균_Silhouette": (
            housing_silhouette_values[mask].mean()
        ),
        "음수비율_%": (
            housing_silhouette_values[mask] < 0
        ).mean() * 100,
    })


housing_silhouette_df = pd.DataFrame(
    housing_silhouette_results
)

display(
    housing_silhouette_df.round(3)
)

,cluster,단지수,단지비율_%,평균_Silhouette,음수비율_%
0,0,2757,36.367,0.277,2.793
1,1,3783,49.901,0.400,0.000
2,2,1041,13.732,0.229,14.697


In [17]:
# ============================================================
# 주택특성 군집 프로파일
# ============================================================

housing_profile_cols = [
    "전용면적",
    "전용면적_IQR",
    "기준연식",
    "층",
]

housing_cluster_profile = (
    complex_df
    .groupby("housing_cluster")[housing_profile_cols]
    .median()
)

housing_cluster_profile["단지수"] = (
    complex_df
    .groupby("housing_cluster")
    .size()
)

housing_cluster_profile["단지비율_%"] = (
    housing_cluster_profile["단지수"]
    / len(complex_df)
    * 100
)

display(
    housing_cluster_profile.round(2)
)

,전용면적,전용면적_IQR,기준연식,층,단지수,단지비율_%
housing_cluster,,,,,,
0,82.45,24.71,23.0,8.0,2757,36.37
1,84.15,0.00,22.0,5.0,3783,49.90
2,36.77,0.48,7.0,9.0,1041,13.73


In [19]:
# ============================================================
# 주택특성 군집 라벨
# ============================================================

housing_cluster_name_map = {
    0: "다평형 일반형",
    1: "중형 표준형",
    2: "소형·신축형",
}

complex_df["housing_cluster_name"] = (
    complex_df["housing_cluster"]
    .map(housing_cluster_name_map)
)

display(
    complex_df[
        [
            "단지일련번호",
            "단지명",
            "location_cluster_name",
            "housing_cluster_name",
        ]
    ]
    .head()
)

,단지일련번호,단지명,location_cluster_name,housing_cluster_name
0,11110-101,크래스빌,일반 주거형,다평형 일반형
1,11110-102,현대뜨레비앙,직주근접 도심형,중형 표준형
2,11110-107,경희궁파크팰리스,직주근접 도심형,다평형 일반형
3,11110-111,창림,직주근접 도심형,중형 표준형
4,11110-112,신현(102동),일반 주거형,중형 표준형


### 주택특성 군집 해석

KMeans K=3을 기준으로 서울 아파트 단지의 구조적 특성을
다음 세 가지 유형으로 구분하였다.

| Cluster | 유형 | 주요 특징 |
|---|---|---|
| 0 | 다평형 일반형 | 중형 면적 중심이며 단지 내 평형 구성이 다양함 |
| 1 | 중형 표준형 | 약 84㎡ 중심의 가장 일반적인 중형 아파트 유형 |
| 2 | 소형·신축형 | 면적이 작고 연식이 짧은 신축 소형 주거 유형 |

- **다평형 일반형**은 전용면적 중앙값은 중형 수준이지만
  면적 IQR이 가장 커 다양한 평형으로 구성된 특징이 있다.
- **중형 표준형**은 전체 단지의 약 절반을 차지하며,
  전용 84㎡ 수준의 표준적인 주거 단지 성격을 보인다.
- **소형·신축형**은 약 37㎡의 소형 면적과 7년 수준의 짧은 연식을 보여
  다른 두 군집과 뚜렷한 구조적 차이를 보인다.

주택특성 군집은 입지 군집과 독립적으로 구성되어,
하나의 아파트 단지를 `입지 유형 + 주택특성 유형`의 두 축으로 설명할 수 있다.

## 5. Clustering V1 최종 요약

### 최종 구성

| 구분 | 입지 Segmentation | 주택특성 Segmentation |
|---|---|---|
| 단위 | 아파트 단지 | 아파트 단지 |
| 알고리즘 | KMeans | KMeans |
| K | 3 | 3 |
| Feature | 역세권거리, 주요일자리거리 | 전용면적, 전용면적 IQR, 기준연식, 층 |
| Silhouette | **0.3725 (전체 평가)** | **0.3320** |
| 활용 | 교통·직주근접 입지 유형 | 단지 구조·연식·평형 유형 |

### 최종 판단

1. **가격은 군집 Feature에 사용하지 않았다.**
   - 거래금액이나 평당가는 군집을 정의하지 않으며, 필요 시 사후 분석에만 사용한다.

2. **통합 Feature보다 두 개의 독립적인 축이 더 잘 작동했다.**
   - 입지와 주택구조를 한 모델에 모두 넣었을 때 군집 경계가 흐려졌다.
   - 반대로 `교통·업무`와 `주택구조`를 분리하면 각 유형의 의미가 선명해졌다.

3. **V1에서는 두 모델 모두 K=3으로 통일한다.**
   - K=2는 분리도가 높더라도 서비스상 이분 구조가 지나치게 단순했다.
   - 주택특성 K=5는 일부 군집의 불안정성이 커 과도한 세분화로 판단했다.

4. **서비스에서는 한 단지에 두 개의 태그를 제공할 수 있다.**
   - 예: `직주근접 도심형` + `주택특성 유형`
   - 입지 유형과 주택특성 유형을 독립적으로 보여주기 때문에 설명력이 높다.


### 최종 단지 유형

Clustering V1에서는 하나의 통합 군집이 아니라
서로 다른 의미를 갖는 두 개의 독립적인 Segmentation을 사용한다.

#### 입지 유형
- 역세권 생활주거형
- 직주근접 도심형
- 일반 주거형

#### 주택특성 유형
- 다평형 일반형
- 중형 표준형
- 소형·신축형

따라서 하나의 단지를 다음과 같이 두 개의 축으로 설명할 수 있다.

> 예시  
> 입지 유형: 직주근접 도심형  
> 주택특성 유형: 소형·신축형

가격은 군집 Feature에 포함하지 않았으며,
군집 결과는 향후 가격예측 결과의 해석 및 서비스상 단지 설명에 활용한다.

In [20]:
# ============================================================
# Test 단지 Coverage 확인
# ============================================================

train_complex_ids = set(
    complex_df["단지일련번호"]
)

test_complex_ids = set(
    cluster_test_df["단지일련번호"]
)


seen_complex_ids = (
    test_complex_ids
    & train_complex_ids
)

unseen_complex_ids = (
    test_complex_ids
    - train_complex_ids
)


print("Test 전체 단지:", f"{len(test_complex_ids):,}")
print("Seen 단지:", f"{len(seen_complex_ids):,}")
print("Unseen 단지:", f"{len(unseen_complex_ids):,}")

print(
    "Seen 단지 Coverage:",
    f"{len(seen_complex_ids) / len(test_complex_ids) * 100:.2f}%",
)

Test 전체 단지: 5,038
Seen 단지: 4,748
Unseen 단지: 290
Seen 단지 Coverage: 94.24%


In [21]:
# ============================================================
# 2026 신규 단지 Feature 생성
# ============================================================

unseen_test_df = (
    cluster_test_df[
        cluster_test_df["단지일련번호"]
        .isin(unseen_complex_ids)
    ]
    .copy()
)


unseen_complex_df = (
    unseen_test_df
    .groupby("단지일련번호")
    .agg(
        단지명=("단지명", "first"),
        법정동=("법정동", "first"),

        전용면적=("전용면적", "median"),
        전용면적_Q25=(
            "전용면적",
            lambda x: x.quantile(0.25),
        ),
        전용면적_Q75=(
            "전용면적",
            lambda x: x.quantile(0.75),
        ),

        건축년도=("건축년도", "median"),
        층=("층", "median"),

        역세권거리_m=("역세권거리_m", "median"),
        주요일자리거리_m=(
            "주요일자리거리_m",
            "median",
        ),
    )
    .reset_index()
)


unseen_complex_df["전용면적_IQR"] = (
    unseen_complex_df["전용면적_Q75"]
    - unseen_complex_df["전용면적_Q25"]
)


# Train과 동일한 기준연도 사용
unseen_complex_df["기준연식"] = (
    2025 - unseen_complex_df["건축년도"]
).clip(lower=0)


print(
    "신규 단지:",
    f"{len(unseen_complex_df):,}",
)

display(
    unseen_complex_df.head()
)

신규 단지: 290


,단지일련번호,단지명,법정동,전용면적,전용면적_Q25,전용면적_Q75,건축년도,층,역세권거리_m,주요일자리거리_m,전용면적_IQR,기준연식
0,11110-105,탑스빌,숭인동,55.040,55.04,55.040,2003.0,6.0,325.770223,2667.187563,0.000,22.0
1,11110-2073,형우럭스빌(33-1),평창동,103.070,103.07,103.070,2005.0,2.5,3108.451556,3879.039995,0.000,20.0
2,11110-2128,건양하늘터,명륜1가,84.780,84.78,84.780,2006.0,4.0,712.373310,1576.007129,0.000,19.0
3,11110-217,(627-120),창신동,45.040,45.04,45.040,1999.0,5.0,481.440616,1684.344531,0.000,26.0
4,11110-3467,그린나래,평창동,36.785,36.72,36.865,2026.0,4.0,2920.936765,3953.814778,0.145,0.0


### 신규 단지의 연식 기준

주택특성 군집 모델은 2025년 말까지의 데이터를 이용해
`2025 - 건축년도`를 기준연식으로 학습하였다.

따라서 2026년 신규 단지에도 동일한 Feature 정의를 적용하여
Train과 Test의 입력 기준을 일치시켰다.

In [22]:
# ============================================================
# Test용 Feature 변환
# ============================================================

def transform_cluster_data(
    data,
    feature_cols,
    scaler,
):

    model_df = data[feature_cols].copy()

    for col in feature_cols:

        if col in LOG_COLS:

            model_df[col] = np.log1p(
                model_df[col].clip(lower=0)
            )

    return scaler.transform(
        model_df
    )

In [23]:
# ============================================================
# 신규 단지 입지 군집 예측
# ============================================================

X_unseen_location = transform_cluster_data(
    unseen_complex_df,
    LOCATION_FEATURES,
    location_scaler,
)


unseen_complex_df["location_cluster"] = (
    location_model.predict(
        X_unseen_location
    )
)


unseen_complex_df["location_cluster_name"] = (
    unseen_complex_df["location_cluster"]
    .map(location_cluster_name_map)
)

In [24]:
# ============================================================
# 신규 단지 주택특성 군집 예측
# ============================================================

X_unseen_housing = transform_cluster_data(
    unseen_complex_df,
    HOUSING_FEATURES,
    housing_scaler,
)


unseen_complex_df["housing_cluster"] = (
    housing_model.predict(
        X_unseen_housing
    )
)


unseen_complex_df["housing_cluster_name"] = (
    unseen_complex_df["housing_cluster"]
    .map(housing_cluster_name_map)
)


display(
    unseen_complex_df[
        [
            "단지일련번호",
            "단지명",
            "location_cluster_name",
            "housing_cluster_name",
        ]
    ]
    .head(10)
)

,단지일련번호,단지명,location_cluster_name,housing_cluster_name
0,11110-105,탑스빌,직주근접 도심형,중형 표준형
1,11110-2073,형우럭스빌(33-1),일반 주거형,중형 표준형
2,11110-2128,건양하늘터,직주근접 도심형,중형 표준형
3,11110-217,(627-120),직주근접 도심형,중형 표준형
4,11110-3467,그린나래,일반 주거형,소형·신축형
5,11110-95,MID그린(9동),직주근접 도심형,중형 표준형
6,11140-1013,노블레스하우스,역세권 생활주거형,중형 표준형
7,11140-1248,(1546-0),직주근접 도심형,소형·신축형
8,11140-30,동평화패션타운,직주근접 도심형,중형 표준형
9,11140-63,청구파크뷰,직주근접 도심형,중형 표준형


In [25]:
# ============================================================
# 단지별 최종 Cluster Lookup
# ============================================================

cluster_lookup = (
    complex_df[
        [
            "단지일련번호",
            "단지명",
            "location_cluster",
            "location_cluster_name",
            "housing_cluster",
            "housing_cluster_name",
        ]
    ]
    .copy()
)


unseen_lookup = (
    unseen_complex_df[
        [
            "단지일련번호",
            "단지명",
            "location_cluster",
            "location_cluster_name",
            "housing_cluster",
            "housing_cluster_name",
        ]
    ]
    .copy()
)


cluster_lookup = (
    pd.concat(
        [
            cluster_lookup,
            unseen_lookup,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        "단지일련번호"
    )
)


print(
    "Cluster Lookup 단지:",
    f"{len(cluster_lookup):,}",
)

Cluster Lookup 단지: 7,871


In [26]:
# ============================================================
# Test 거래에 군집 Feature 연결
# ============================================================

test_clustered_df = (
    cluster_test_df
    .merge(
        cluster_lookup[
            [
                "단지일련번호",
                "location_cluster",
                "location_cluster_name",
                "housing_cluster",
                "housing_cluster_name",
            ]
        ],
        on="단지일련번호",
        how="left",
    )
)


display(
    test_clustered_df[
        [
            "transaction_id",
            "단지명",
            "location_cluster_name",
            "housing_cluster_name",
        ]
    ]
    .head()
)

,transaction_id,단지명,location_cluster_name,housing_cluster_name
0,1740,현대뜨레비앙,직주근접 도심형,중형 표준형
1,1741,삼전솔하임4차,역세권 생활주거형,소형·신축형
2,1742,동대문,역세권 생활주거형,중형 표준형
3,1743,아남1,직주근접 도심형,중형 표준형
4,1744,아남1,직주근접 도심형,중형 표준형


In [27]:
# ============================================================
# Test Coverage
# ============================================================

coverage_summary = pd.DataFrame({
    "항목": [
        "Test 전체 단지",
        "Train Seen 단지",
        "Train Unseen 단지",
        "Test 전체 거래",
        "군집 할당 거래",
    ],

    "값": [
        len(test_complex_ids),
        len(seen_complex_ids),
        len(unseen_complex_ids),
        len(test_clustered_df),

        test_clustered_df[
            "location_cluster"
        ].notna().sum(),
    ],
})


complex_seen_coverage = (
    len(seen_complex_ids)
    / len(test_complex_ids)
    * 100
)

transaction_assignment_coverage = (
    test_clustered_df[
        "location_cluster"
    ]
    .notna()
    .mean()
    * 100
)


print(
    "기존 단지 Coverage:",
    f"{complex_seen_coverage:.2f}%",
)

print(
    "최종 거래 군집 할당률:",
    f"{transaction_assignment_coverage:.2f}%",
)

display(
    coverage_summary
)

기존 단지 Coverage: 94.24%
최종 거래 군집 할당률: 100.00%


,항목,값
0,Test 전체 단지,5038
1,Train Seen 단지,4748
2,Train Unseen 단지,290
3,Test 전체 거래,39569
4,군집 할당 거래,39569


In [29]:
from pathlib import Path

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "clustering"
    / "v1"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


In [30]:
cluster_lookup.to_parquet(
    MODEL_DIR / "complex_clusters.parquet",
    index=False,
)


location_cluster_profile.to_csv(
    MODEL_DIR / "location_cluster_profile.csv",
    encoding="utf-8-sig",
)


housing_cluster_profile.to_csv(
    MODEL_DIR / "housing_cluster_profile.csv",
    encoding="utf-8-sig",
)


cluster_comparison_df.to_csv(
    MODEL_DIR / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)


print("저장 완료:")
print(MODEL_DIR)

저장 완료:
c:\Users\uk\Apartment price\models\clustering\v1


### Test Coverage 및 신규 단지 적용 결과

2026년 상반기 Test 데이터에는 총 **5,038개 단지**가 존재하였다.

이 중 **4,748개(94.24%)**는 학습기간에 이미 존재했던 단지이며,
**290개(5.76%)**는 2026년 Test 기간에 처음 등장한 신규 단지였다.

기존 단지는 학습 단계에서 부여된 군집을 그대로 유지하였으며,
신규 단지는 Train 데이터로 학습한 Scaler와 KMeans 모델을 사용하여
동일한 Feature 기준으로 군집을 예측하였다.

그 결과 Test 전체 **39,569건의 거래에 군집 정보가 모두 할당되어
최종 거래 기준 Coverage 100%를 확보하였다.**

이는 향후 서비스 환경에서 새로운 단지가 등장하더라도
기존 군집 체계를 유지하면서 입지 유형과 주택특성 유형을
자동으로 부여할 수 있음을 의미한다.


### V2 고도화 후보

- 학교알리미 기반 초등학교 교육환경 점수
- 우수 교육환경 학교 접근성
- 추가 입지·생활 인프라 고도화

V2에서는 데이터를 추가하되,
V1의 입지·주택특성 2축 구조와 비교해 실제 군집 품질이 개선되는지 검증한다.